# Analisis Klaster Daerah Rawan Perceraian Berdasarkan Indikator Kesejahteraan Masyarakat

**MVP Portfolio - Jawa Timur**

Notebook ini menjalankan pipeline sederhana: load data, feature engineering, missing value handling, standardisasi fitur, K-Means clustering, evaluasi silhouette score, EDA, dan peta interaktif Folium.

> Catatan: file sample di repository adalah data sintetis untuk mengetes alur notebook. Untuk portfolio final, gunakan data resmi BPS/Pengadilan Agama dan GeoJSON/shapefile resmi.

## Step 0: Install dan Import Library

Jalankan cell instalasi ini di Google Colab. Jika berjalan lokal dan library sudah tersedia, cell ini boleh dilewati.

In [ ]:
# Colab biasanya belum menyertakan semua library geospatial terbaru.
!pip -q install geopandas folium mapclassify openpyxl

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

import geopandas as gpd
import folium
from folium.features import GeoJsonTooltip

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")

## Step 1: Konfigurasi Path

Sesuaikan path berikut setelah upload file ke Colab. Untuk tes awal, gunakan sample CSV dari repository.

In [ ]:
# Jika notebook dibuka dari repo GitHub di Colab, jalankan dari root repo.
# Jika file belum ada di Colab, upload manual lewat sidebar Files.
BASE_DIR = Path(".")
DATA_PATH = BASE_DIR / "data" / "sample_jatim_indicators.csv"

# Ganti dengan path GeoJSON/shapefile Anda.
# Contoh: GEO_PATH = Path("data/raw/jatim_kabkota.geojson")
# Contoh shapefile ZIP: GEO_PATH = Path("data/raw/shapefile_jatim.zip")
GEO_PATH = None

# Kolom kunci pada GeoJSON/shapefile yang sepadan dengan kode_wilayah.
# Contoh umum: "kode_wilayah", "KODE_KAB", "KODE_BPS", "ADM2_PCODE".
GEOJSON_KEY = "kode_wilayah"

OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
MAP_OUTPUT = OUTPUT_DIR / "map_cluster.html"

FEATURE_COLUMNS = ["ipm", "tpt", "kemiskinan", "tingkat_cerai"]
N_CLUSTERS = 3
RANDOM_STATE = 42

## Step 2: Load Data Tabular

Notebook mendukung `.csv`, `.xlsx`, dan `.xls`. Nama kolom akan dinormalisasi menjadi huruf kecil dan snake_case agar lebih stabil.

In [ ]:
def normalize_column_name(column):
    return (
        str(column)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("-", "_")
    )


def load_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {path}. Upload data atau sesuaikan DATA_PATH.")

    if path.suffix.lower() == ".csv":
        df_loaded = pd.read_csv(path)
    elif path.suffix.lower() in [".xlsx", ".xls"]:
        df_loaded = pd.read_excel(path)
    else:
        raise ValueError("Format data harus .csv, .xlsx, atau .xls")

    df_loaded.columns = [normalize_column_name(col) for col in df_loaded.columns]
    return df_loaded


df = load_table(DATA_PATH)
df.head()

In [ ]:
df.info()

## Step 3: Validasi Kolom dan Feature Engineering

Kolom wajib: `kode_wilayah`, `kabupaten_kota`, `ipm`, `tpt`, `kemiskinan`, `jumlah_nikah`, `jumlah_cerai`.

In [ ]:
REQUIRED_COLUMNS = [
    "kode_wilayah",
    "kabupaten_kota",
    "ipm",
    "tpt",
    "kemiskinan",
    "jumlah_nikah",
    "jumlah_cerai",
]

missing_columns = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
if missing_columns:
    raise ValueError(f"Kolom wajib belum ada: {missing_columns}")

df = df.copy()
df["kode_wilayah"] = df["kode_wilayah"].astype(str).str.strip()

numeric_columns = ["ipm", "tpt", "kemiskinan", "jumlah_nikah", "jumlah_cerai"]
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["tingkat_cerai"] = np.where(
    df["jumlah_nikah"] > 0,
    (df["jumlah_cerai"] / df["jumlah_nikah"]) * 100,
    np.nan,
)

df[["kode_wilayah", "kabupaten_kota", *FEATURE_COLUMNS]].head()

## Step 4: Handling Missing Value

Untuk MVP, missing value pada fitur numerik diisi dengan rata-rata kolom. Pada versi artikel ilmiah, strategi imputasi perlu dijelaskan lebih hati-hati.

In [ ]:
df[FEATURE_COLUMNS].isnull().sum()

In [ ]:
for col in FEATURE_COLUMNS:
    df[col] = df[col].fillna(df[col].mean())

df[FEATURE_COLUMNS].isnull().sum()

## Step 5: EDA Visual

Heatmap membantu membaca hubungan antarindikator. Scatter plot memberi gambaran kasar pola antara kemiskinan dan tingkat perceraian.

In [ ]:
plt.figure(figsize=(8, 6))
corr = df[FEATURE_COLUMNS].corr()
sns.heatmap(corr, annot=True, cmap="RdBu_r", center=0, fmt=".2f", linewidths=0.5)
plt.title("Korelasi Indikator Kesejahteraan dan Tingkat Cerai")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="kemiskinan", y="tingkat_cerai", s=90)
plt.title("Kemiskinan vs Tingkat Cerai")
plt.xlabel("Kemiskinan (%)")
plt.ylabel("Tingkat Cerai per 100 Pernikahan")
plt.tight_layout()
plt.show()

## Step 6: Standardisasi Fitur

K-Means sensitif terhadap skala. Karena IPM, TPT, kemiskinan, dan tingkat cerai punya rentang berbeda, semua fitur distandardisasi dengan `StandardScaler`.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[FEATURE_COLUMNS])

scaled_df = pd.DataFrame(X_scaled, columns=[f"scaled_{col}" for col in FEATURE_COLUMNS])
scaled_df.head()

## Step 7: K-Means Clustering dan Silhouette Score

Untuk MVP, jumlah cluster ditetapkan 3 sesuai scope. Silhouette score dipakai sebagai indikator kualitas pemisahan cluster.

In [ ]:
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

score = silhouette_score(X_scaled, df["cluster"])
print(f"Silhouette Score untuk {N_CLUSTERS} cluster: {score:.3f}")

df["cluster"] = df["cluster"].astype(int)
df[["kode_wilayah", "kabupaten_kota", "cluster", *FEATURE_COLUMNS]].head()

In [ ]:
cluster_profile = (
    df.groupby("cluster")[FEATURE_COLUMNS]
    .mean()
    .round(2)
    .sort_values("tingkat_cerai", ascending=False)
)

cluster_profile

In [ ]:
risk_order = cluster_profile.index.tolist()
risk_labels = {
    risk_order[0]: "Kerawanan Tinggi",
    risk_order[1]: "Kerawanan Sedang",
    risk_order[2]: "Kerawanan Rendah",
}

df["label_kerawanan"] = df["cluster"].map(risk_labels)
df[["kabupaten_kota", "cluster", "label_kerawanan", "tingkat_cerai"]].sort_values("tingkat_cerai", ascending=False).head(10)

## Step 8: Visualisasi Cluster Non-Spasial

Scatter plot ini memakai label cluster agar pola hasil K-Means lebih mudah dibaca sebelum masuk ke peta.

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df,
    x="kemiskinan",
    y="tingkat_cerai",
    hue="label_kerawanan",
    style="label_kerawanan",
    s=110,
)
plt.title("Cluster Wilayah Berdasarkan Kemiskinan dan Tingkat Cerai")
plt.xlabel("Kemiskinan (%)")
plt.ylabel("Tingkat Cerai per 100 Pernikahan")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

## Step 9: Load GeoJSON/Shapefile dan Merge Data

Jika belum punya file peta, lewati sementara cell ini. Setelah GeoJSON/shapefile tersedia, set `GEO_PATH` dan `GEOJSON_KEY` pada Step 1.

In [ ]:
def normalize_geo_key(series):
    return series.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)


if GEO_PATH is None:
    print("GEO_PATH belum diisi. Set GEO_PATH ke file .geojson, .json, .shp, atau .zip untuk membuat peta.")
    gdf_merged = None
else:
    gdf = gpd.read_file(GEO_PATH)
    gdf.columns = [normalize_column_name(col) for col in gdf.columns]
    geo_key = normalize_column_name(GEOJSON_KEY)

    if geo_key not in gdf.columns:
        raise ValueError(f"Kolom key '{geo_key}' tidak ditemukan di data peta. Kolom tersedia: {list(gdf.columns)}")

    gdf[geo_key] = normalize_geo_key(gdf[geo_key])
    df["kode_wilayah"] = normalize_geo_key(df["kode_wilayah"])

    gdf_merged = gdf.merge(df, left_on=geo_key, right_on="kode_wilayah", how="left")
    print("Jumlah wilayah di peta:", len(gdf))
    print("Jumlah wilayah berhasil merge:", gdf_merged["cluster"].notna().sum())
    display(gdf_merged[[geo_key, "kabupaten_kota", "cluster", "label_kerawanan"]].head())

## Step 10: Membuat Peta Interaktif Folium

Peta disimpan sebagai HTML agar bisa dibuka di browser dan ditautkan dari README/GitHub Pages.

In [ ]:
COLOR_MAP = {
    "Kerawanan Tinggi": "#d73027",
    "Kerawanan Sedang": "#fee08b",
    "Kerawanan Rendah": "#1a9850",
}


def style_function(feature):
    label = feature["properties"].get("label_kerawanan")
    return {
        "fillColor": COLOR_MAP.get(label, "#bdbdbd"),
        "color": "#333333",
        "weight": 0.7,
        "fillOpacity": 0.78,
    }


if gdf_merged is None:
    print("Peta belum dibuat karena GEO_PATH belum diisi.")
else:
    gdf_map = gdf_merged.to_crs(epsg=4326).copy()
    center_lat = gdf_map.geometry.centroid.y.mean()
    center_lon = gdf_map.geometry.centroid.x.mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles="CartoDB positron")

    tooltip_fields = [
        "kabupaten_kota",
        "label_kerawanan",
        "ipm",
        "tpt",
        "kemiskinan",
        "tingkat_cerai",
    ]
    tooltip_aliases = [
        "Wilayah:",
        "Cluster:",
        "IPM:",
        "TPT (%):",
        "Kemiskinan (%):",
        "Tingkat Cerai:",
    ]

    folium.GeoJson(
        gdf_map,
        name="Cluster Kerawanan Perceraian",
        style_function=style_function,
        tooltip=GeoJsonTooltip(fields=tooltip_fields, aliases=tooltip_aliases, localize=True),
    ).add_to(m)

    legend_html = """
    <div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999; background: white; padding: 12px 14px; border: 1px solid #999; border-radius: 6px; font-size: 13px; box-shadow: 0 2px 8px rgba(0,0,0,0.15);">
      <b>Cluster Kerawanan</b><br>
      <span style="display:inline-block;width:10px;height:10px;background:#d73027;margin-right:6px;"></span>Tinggi<br>
      <span style="display:inline-block;width:10px;height:10px;background:#fee08b;margin-right:6px;"></span>Sedang<br>
      <span style="display:inline-block;width:10px;height:10px;background:#1a9850;margin-right:6px;"></span>Rendah<br>
      <span style="display:inline-block;width:10px;height:10px;background:#bdbdbd;margin-right:6px;"></span>Tidak ada data
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))
    folium.LayerControl(collapsed=False).add_to(m)

    m.save(MAP_OUTPUT)
    print(f"Peta berhasil disimpan ke: {MAP_OUTPUT}")
    display(m)

## Step 11: Export Hasil Cluster

CSV ini bisa dipakai untuk dokumentasi README atau analisis lanjutan.

In [ ]:
cluster_output_path = OUTPUT_DIR / "cluster_result.csv"
df[["kode_wilayah", "kabupaten_kota", "cluster", "label_kerawanan", *FEATURE_COLUMNS]].to_csv(cluster_output_path, index=False)
print(f"Hasil cluster disimpan ke: {cluster_output_path}")

## Ringkasan Portfolio

Tuliskan 3-5 insight utama setelah memakai data resmi, misalnya:

1. Cluster dengan tingkat cerai tertinggi memiliki rata-rata indikator tertentu yang menonjol.
2. Wilayah perkotaan dan perdesaan dapat menunjukkan pola yang berbeda.
3. Hasil cluster bersifat eksploratif, bukan bukti kausal bahwa indikator kesejahteraan menyebabkan perceraian.